# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kevinwdt/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## 1. My lane as an ML task

My lane is **Lane 4: CTR / Engagement Opportunity Scoring**.

I will frame this primarily as a **scoring and ranking task**. The system will estimate how much a page's measured CTR falls below the CTR expected for pages in a similar search-position range. It will then rank pages from highest to lowest review priority.

This is more appropriate than classification because my main question is not simply whether a page is good or bad. The decision is which pages an SEO specialist or content editor should investigate first. The final output will be a ranked review queue rather than an automatic decision to change a page.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "kevinwdt/flyrank-internship-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Use pages with enough impressions and valid positions.
lane_df = df[
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
].copy()

# Compare pages only within similar position ranges.
lane_df["position_band"] = pd.cut(
    lane_df["avg_position"],
    bins=[0, 3, 10, 20],
    labels=["1-3", "4-10", "11-20"]
)

print("ML task type: scoring and ranking")
print("Full starter dataset rows:", f"{len(df):,}")
print("Lane 4 slice rows:", f"{len(lane_df):,}")


ML task type: scoring and ranking
Full starter dataset rows: 30,000
Lane 4 slice rows: 12,023


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

My provisional proxy will be a page's **CTR gap**.

For each search-position band, I will calculate the typical measured CTR. The proxy will then compare each page's measured CTR with the typical CTR for its position band:

**CTR gap = expected CTR for the position band − measured page CTR**

A larger positive gap means that a page is receiving a lower CTR than similar-position pages and may deserve earlier review.

This is a defined proxy, not a confirmed real-world outcome. It does not prove that the page has a bad title or that changing the page will increase clicks. It only creates a reasonable starting score for prioritizing human review.

For a stronger final system, expected CTR should be estimated without allowing a row to help define its own expectation, such as through held-out or cross-validated predictions.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Use the median because it is less affected by extreme values.
lane_df["expected_ctr_proxy"] = (
    lane_df.groupby("position_band", observed=True)["ctr"]
    .transform("median")
)

# Positive values mean the page is below its position band's median CTR.
lane_df["ctr_gap_proxy"] = (
    lane_df["expected_ctr_proxy"] - lane_df["ctr"]
)

target_preview = lane_df[
    [
        "impressions_90d",
        "clicks_90d",
        "avg_position",
        "position_band",
        "ctr",
        "expected_ctr_proxy",
        "ctr_gap_proxy"
    ]
].sort_values("ctr_gap_proxy", ascending=False)

print("Provisional target/proxy column: ctr_gap_proxy")
display(target_preview.head(10))


Provisional target/proxy column: ctr_gap_proxy


,impressions_90d,clicks_90d,avg_position,position_band,ctr,expected_ctr_proxy,ctr_gap_proxy
29924,1086,0,7.3,4-10,0.0,0.24,0.24
94,7737,0,5.5,4-10,0.0,0.24,0.24
29886,742,0,3.7,4-10,0.0,0.24,0.24
4481,1023,0,8.1,4-10,0.0,0.24,0.24
4533,4955,0,3.9,4-10,0.0,0.24,0.24
18989,535,0,3.1,4-10,0.0,0.24,0.24
18942,1139,0,7.7,4-10,0.0,0.24,0.24
18909,1464,0,3.6,4-10,0.0,0.24,0.24
18715,714,0,4.3,4-10,0.0,0.24,0.24
1793,628,0,9.8,4-10,0.0,0.24,0.24


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

My primary success metric will be **Precision@20**.

Precision@20 measures how many of the first 20 recommended pages are confirmed as useful review candidates after human examination.

For example, if 16 of the top 20 recommendations are judged to deserve metadata, intent, content, engagement, or monitoring review, then Precision@20 is 16 divided by 20, or 0.80.

My provisional success goal is **Precision@20 of at least 0.75**. This means that at least 15 of the first 20 recommendations should be useful review candidates.

This target is provisional because the final definition of a useful recommendation should be agreed with the people reviewing the pages. I will not calculate a final Precision@20 until I have an independent outcome or human-reviewed relevance label. Calculating it against the same proxy used to create the ranking would give a misleading result.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review_capacity = 20
precision_goal = 0.75

minimum_useful_recommendations = int(
    np.ceil(review_capacity * precision_goal)
)

print("Review capacity:", review_capacity, "pages")
print("Precision@20 goal:", precision_goal)
print(
    "Minimum useful recommendations needed:",
    minimum_useful_recommendations,
    "out of",
    review_capacity
)

Review capacity: 20 pages
Precision@20 goal: 0.75
Minimum useful recommendations needed: 15 out of 20


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized webpage or content item**.

Each row contains measured information about one page, including its impressions, clicks, CTR, average search position, engagement, content type, intent, age, and freshness.

For Lane 4, I am initially limiting the data to pages with at least 500 impressions and an average position between 1 and 20. This removes pages with very little exposure and pages without a valid or useful search position.

The dataframe below shows the type of information that could be used to score each page. The page identifiers will not be used as model features.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unit_of_analysis = lane_df[
    [
        "content_type",
        "main_intent",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "position_band",
        "engagement_rate",
        "days_since_last_update",
        "ctr_gap_proxy"
    ]
].copy()

print("One row = one pseudonymized webpage/content item")
print("Rows in Lane 4 dataframe:", f"{len(unit_of_analysis):,}")

display(unit_of_analysis.head(10))


One row = one pseudonymized webpage/content item
Rows in Lane 4 dataframe: 12,023


,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,position_band,engagement_rate,days_since_last_update,ctr_gap_proxy
0,keyword article,transactional,3803,29,0.76,10.6,11-20,5.88,20,-0.59
3,keyword article,commercial,11751,58,0.49,6.2,4-10,1.28,22,-0.25
5,keyword article,transactional,3970,1,0.03,8.5,4-10,0.00,20,0.21
9,keyword article,informational,1240,2,0.16,4.9,4-10,0.00,104,0.08
10,keyword article,commercial,20919,324,1.55,2.2,1-3,6.75,104,-1.35
12,keyword article,informational,7228,127,1.76,5.6,4-10,3.43,104,-1.52
16,keyword article,informational,13848,21,0.15,8.9,4-10,0.21,104,0.09
17,keyword article,transactional,9449,7,0.07,7.3,4-10,11.86,22,0.17
18,keyword article,transactional,5141,7,0.14,11.4,11-20,2.33,20,0.03
20,keyword article,informational,2538,14,0.55,5.6,4-10,7.69,20,-0.31


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML may beat a fixed rule here

A fixed rule could say that every page with CTR below 0.5% should be reviewed. However, this rule ignores the fact that expected CTR changes with average position. It may also ignore impressions, content type, intent, freshness, engagement, and other interacting signals.

For example, a CTR that is weak for a page in position 2 may be normal for a page in position 18. A single global threshold could therefore recommend too many pages or rank them in an unhelpful order.

Machine learning may help by estimating expected CTR from several interacting signals and identifying pages that perform below that expectation. It could also learn patterns that are difficult to express with one short if-statement.

However, ML has not earned its place automatically. I will first create a transparent position-adjusted rule as a baseline. A model should only be used if it produces a more useful and reliable top-ranked review list than that baseline.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Demonstrate the weakness of one global CTR threshold.
fixed_threshold = 0.5  # This means 0.5%, not 50%.

rule_comparison = (
    lane_df.groupby("position_band", observed=True)
    .agg(
        pages=("ctr", "size"),
        median_ctr=("ctr", "median"),
        pages_flagged=(
            "ctr",
            lambda values: (values < fixed_threshold).sum()
        )
    )
)

rule_comparison["percentage_flagged"] = (
    rule_comparison["pages_flagged"]
    / rule_comparison["pages"]
    * 100
)

print("Example fixed rule: flag every page with CTR below 0.5%")
display(rule_comparison.round(2))

Example fixed rule: flag every page with CTR below 0.5%


,pages,median_ctr,pages_flagged,percentage_flagged
position_band,,,,
1-3,480,0.20,360,75.00
4-10,7084,0.24,5609,79.18
11-20,4459,0.17,3790,85.00


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.